# **Note**: To see the details that main_notebook.ipynb didn't talk about jump right to 1.3.1 - 1.3.3

### Where should I park near a target place?

We want to use the City of Pittsburgh parking datasets to rank and recommend nearby parking zones, helping drivers choose parking more efficiently and reduce time spent searching for a spot.

This is really cool because you can plug in any destination (any “X,Y” point) in Pittsburgh, and the model will generate a personalized parking recommendation based on your preferences — so the “best” option adapts to what you care about most.

### 1.1. Method
We solve this problem along three dimensions: **price**, **availability**, and **distance**, and the final recommendation is a **ranked list of parking options** (location) near the destination, ordered by a single preference-weighted score.

To compute these dimensions for each nearby parking meter, we first derive a few practical variables—meter **location**, **capacity**, **active hours**, and **observed demand**—from the Pittsburgh parking CSVs:

- **Location (x/y of each meter):** from **parking_meters.csv**  
- **Capacity (spaces per zone):** from **parking_spaces_zone_group_norm.csv**  
- **Price (hourly rate):** from **parking_rate.csv**, joined to meters to assign a rate to each candidate meter  
- **Active hours (when the meter is “on”):** from **parking_hours.csv**, used to filter transactions to valid enforcement periods  
- **Demand / busyness (usage level):** from **payment_transactions.csv**, counting transactions during active hours  
  - then compute an availability proxy such as **transactions_per_space = total_transactions / total_spaces**
- **Distance to destination:** computed from meter coordinates in **parking_meters.csv** using **haversine**

After computing these quantities for all candidate meters near the destination, we normalize them and combine them with user-defined weights into a single score, producing the final ranked parking recommendations.

### 1.2. Constraint / Assumption 

Transaction counts and space capacity are only available at the **zone level**, not per individual meter.  
To estimate meter-level values, we assume demand and capacity are uniformly distributed within each zone.

- **Meter-level spaces:**
$$
\text{spaces\_per\_meter}=\frac{\text{total\_spaces\_in\_zone}}{\text{\# meters in zone}}
$$

- **Meter-level transactions** (within the chosen time window and active hours):
$$
\text{transactions\_per\_meter}=\frac{\text{total\_transactions\_in\_zone}}{\text{\# meters in zone}}
$$

We then compute the busyness proxy at the meter level using these evenly divided estimates.  
(Equivalently, this yields the same ratio as zone-level $\text{transactions\_per\_space}$, but lets us score individual meters consistently.)

### 1.3. Code

**Imports & Helper Function**

We don't have distance information in datasets but we do have x, y coordinates. Thus, we could use Haversine formula to calculate distance. 

Reference: https://en.wikipedia.org/wiki/Haversine_formula

In [1]:
import polars as pl
import altair as alt
import numpy as np

# ----------------
# DATA IMPORTS
# ----------------
parking_rates = pl.read_csv("../final_datasets/parking_rates.csv")
parking_meters = pl.read_csv("../final_datasets/parking_meters.csv")
# parking spaces (zone level)
parking_spaces_zone_group_norm = pl.read_csv("../final_datasets/parking_spaces_zone_group_norm.csv")
# payment_transactions (zone level)
payment_transactions = pl.read_csv("../final_datasets/payment_transactions.csv")
# used to join zone-level results to other parking tables keyed by zone_id
# i.e. parking_xxx.csv 
zone_group_norm_to_zone_id = pl.read_csv("../final_datasets/zone_group_norm_to_zone_id.csv")

# define joinable zone list
joinable_zones_lst = (
    parking_rates.join(zone_group_norm_to_zone_id, on="zone_id")
    # we have one to many zone_group_norm - zone_id mapping. See datasets_join.ipynb for details
    .unique(subset="zone_group_norm",keep="first")
    .join(parking_spaces_zone_group_norm, on="zone_group_norm")
    .join(payment_transactions, on="zone_group_norm")
    # some zones cannot be normalized to allow joining between different datasets
    # so they are not joinable. See datasets_join.ipynb for details
    .select("zone_group_norm")
    .unique().to_series().to_list()
)

# ---------------------------------------------------
# HELPER FUNC: Haversine formula distance calculation
# ---------------------------------------------------
def haversine_m(lon1, lat1, lon2, lat2):
    """
    Compute the great-circle (Haversine) distance between two points on Earth.

    Parameters
    lon1, lat1 : float or array-like
        Longitude and latitude of the first point in decimal degrees.
    lon2, lat2 : float or array-like
        Longitude and latitude of the second point in decimal degrees.

    Returns
    float or np.ndarray
        Great-circle distance between the two points in **meters**.
        The return type matches the input type: scalar inputs return a float,
        array-like inputs return a NumPy array.
    """
    R = 6371000.0
    phi1 = np.radians(lat1)
    phi2 = np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dl = np.radians(lon2 - lon1)

    a = np.sin(dphi/2.0)**2 + np.cos(phi1)*np.cos(phi2)*np.sin(dl/2.0)**2
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1-a))


In [2]:
# --------
# Distance Calculation
# --------
meters = (
    parking_meters
    .join(zone_group_norm_to_zone_id, on="zone_id")
    .filter(
        pl.col("zone_group_norm").is_in(joinable_zones_lst)
    )
)
# target destination's (x,y)
CMU_x, CMU_y = -79.9457643, 40.4443039 

# radius choices (meters)
RADIUS_M = 800   # ~0.5 mile

meters_near = (
    meters
    .with_columns(
        pl.struct(["x", "y"]).map_elements(
            lambda s: float(haversine_m(s["x"], s["y"], CMU_x, CMU_y))
            # lambda s: float(haversine_m(s["x"], s["y"], PANDA_x, PANDA_y))
        ).alias("dist_m")
    )
    .filter(pl.col("dist_m") <= RADIUS_M)
    .sort("dist_m")
)


# ------------------------------------------
# Joining Meters, Rate, Transaction, Spaces
# ------------------------------------------
# build zone totals
zone_totals = (
    payment_transactions
    .group_by("zone_group_norm")
    .agg([
        pl.col("n_transaction").sum().alias("zone_transactions"),
    ])
)

# count meters per zone
meters_per_zone = (
    meters_near
    .select(["zone_group_norm", "meter_id"])
    .unique()
    .group_by("zone_group_norm")
    .len()
    .rename({"len": "n_meters_zone"})
)

# join spaces + zone transactions to meters, then allocate
meters_rates_spaces_tx_near = (
    meters_near.select(['meter_id', 'location', 'x', 'y', 'dist_m','zone_group_norm'])
    .join(parking_rates, on="meter_id")
    .join(parking_spaces_zone_group_norm, on="zone_group_norm", how="left")
    .join(zone_totals, on="zone_group_norm", how="left")
    .join(meters_per_zone, on="zone_group_norm", how="left")
    # allocate zone-level values down to meter level (uniform split assumption)
    .with_columns([
        (pl.col("spaces") / pl.col("n_meters_zone")).alias("spaces_per_meter"),
        (pl.col("zone_transactions") / pl.col("n_meters_zone")).alias("tx_per_meter"),
    ])
    .select([
        'meter_id', 'location', 'x', 'y', 'dist_m', 'zone_group_norm', 
        'rate_window', 'rate_time', 'rate', 'type', 
        'zone_transactions', 'spaces_per_meter', 'tx_per_meter'
    ])
)

We have three type of rate time: all time, 8AM - 2PM and 2PM - 6PM

In [3]:
# rate time check
meters_rates_spaces_tx_near.select("rate_time").unique()

rate_time
str
"""~ 8AM - 2PM"""
"""~ 2PM - 6PM"""
"""all time"""


In [4]:
# Divede dataset to morning (8AM - 2PM) and afternoon (2PM - 6PM)
morning_analysis = (
    meters_rates_spaces_tx_near
    .filter(
        pl.col("rate_time") != "~ 2PM - 6PM"
    )
)
afternoon_analysis = (
    meters_rates_spaces_tx_near
    .filter(
        pl.col("rate_time") != "~ 8AM - 2PM"
    )
)

### 1.3.1. Price Ranking 

In [5]:
def make_rate_ranking(df: pl.DataFrame, period: str) -> pl.DataFrame:
    return (
        df.group_by("rate")
        .agg([
            pl.len().alias("n_rows"),
            pl.col("location").unique().sort().alias("locations_list"),
        ])
        .with_columns([
            pl.col("locations_list").list.join(", ").alias("locations"),
            pl.lit(period).alias("period"),
        ])
        .select(["period", "rate", "n_rows", "locations"])
    )

rate_morning = make_rate_ranking(morning_analysis, "Morning")
rate_afternoon = make_rate_ranking(afternoon_analysis, "Afternoon")

df_both = pl.concat([rate_morning, rate_afternoon]).sort(["period", "rate"]).to_pandas()

df_m = rate_morning.to_pandas()
df_a = rate_afternoon.to_pandas()

c1 = alt.Chart(df_m).mark_bar().encode(
    x=alt.X("rate:O", title="Rate ($/hour)", 
            sort="ascending", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("n_rows:Q", title="Frequency (rows)"),
    color=alt.Color("rate:O", legend=None),
    tooltip=["rate", "n_rows", "locations"],
).properties(width=260, height=300, title="Morning")

c2 = alt.Chart(df_a).mark_bar().encode(
    x=alt.X("rate:O", title="Rate ($/hour)", 
            sort="ascending", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("n_rows:Q", title="Frequency (rows)"),
    color=alt.Color("rate:O", legend=None),
    tooltip=["rate", "n_rows", "locations"],
).properties(width=260, height=300, title="Afternoon")

(c1 | c2).resolve_scale(color="shared",)

alt.HConcatChart(...)

### 1.3.2. Distance Ranking

In [10]:
dist_rank = (
    morning_analysis
    .group_by("location")
    .agg([
        pl.col("x").min().alias("min_x"),
        pl.col("y").min().alias("min_y"),
        pl.col("dist_m").min().alias("min_dist_m"),
        pl.col("dist_m").median().alias("median_dist_m"),
        pl.col("meter_id").n_unique().alias("n_meters"),
    ])
    .sort("min_dist_m")
)

dist_rank

df = dist_rank.to_pandas()

alt.Chart(df.head(10)).mark_bar().encode(
    x=alt.X("min_dist_m:Q", title="Closest meter distance to HBH (m)"),
    y=alt.Y("location:N", sort="x", title="Location"),
    color=alt.Color("min_dist_m:Q"),
    tooltip=[
    "location",
    alt.Tooltip("min_x:Q", format=".5f", title="Closest x"),
    alt.Tooltip("min_y:Q", format=".5f", title="Closest y"),
    alt.Tooltip("min_dist_m:Q", format=".2f", title="Min dist (m)"),
    alt.Tooltip("median_dist_m:Q", format=".2f", title="Median dist (m)"),
    alt.Tooltip("n_meters:Q", title="# meters"),
]
).properties(width=600, height=450, title="Closest meter distance to HBH by location")

alt.Chart(...)

### 1.3.3. Busyness Ranking

In [9]:
busyness_rank = (
    morning_analysis
    .group_by("location")
    .agg([
        pl.col("x").min().alias("min_x"),
        pl.col("y").min().alias("min_y"),
        pl.col("tx_per_meter").sum().alias("tot_transactions"),
        pl.col("spaces_per_meter").sum().alias("tot_spaces"),
        pl.col("dist_m").min().alias("min_dist_m"),
        pl.col("dist_m").median().alias("median_dist_m"),
        pl.col("meter_id").n_unique().alias("n_meters"),
    ])
    .with_columns(
        (pl.col("tot_transactions") / pl.col("tot_spaces")).alias("transactions_per_space")
    )
    .sort("transactions_per_space", descending=True)
)

alt.Chart(busyness_rank.to_pandas()).mark_bar().encode(
    x=alt.X("transactions_per_space:Q", title="Transactions per space"),
    y=alt.Y("location:N", sort="x", title="Location"),
    color=alt.Color("transactions_per_space:Q", title="Transactions/space"),
    tooltip=[
        "location",
        alt.Tooltip("min_x:Q", format=".6f", title="Closest x"),
        alt.Tooltip("min_y:Q", format=".6f", title="Closest y"),
        alt.Tooltip("transactions_per_space:Q", format=".3f", title="Tx per space (busyness)"),
        alt.Tooltip("tot_transactions:Q", format=",.0f", title="Total transactions"),
        alt.Tooltip("tot_spaces:Q", format=",.0f", title="Total spaces"),
        alt.Tooltip("min_dist_m:Q", format=".2f", title="Min dist (m)"),
        alt.Tooltip("median_dist_m:Q", format=".2f", title="Median dist (m)"),
        alt.Tooltip("n_meters:Q", format=",.0f", title="# meters"),
    ]
).properties(
    width=650,
    height=450,
    title="Busyness near HBH: Transactions per space by location (Top 15)"
)

alt.Chart(...)

### 1.3.4. Final Weighted Ranking

In [8]:
# ------------
# HELPER FUNC
# ------------
def location_summary(df, period):
    """
    Aggregate meter-level observations into a location-level summary table.

    Parameters
    df : pl.DataFrame
        Meter-level data containing (at minimum) columns:
        ['location', 'x', 'y', 'rate', 'tx_per_meter', 'spaces_per_meter', 'dist_m', 'meter_id'].
        Each row typically represents a meter observation (or a meter-rate record) near the destination.
    period : str
        Label for the time slice being summarized (e.g., "Morning", "Afternoon").

    Returns
    pl.DataFrame
        One row per `location` with representative price, demand/capacity totals,
        distance summaries, and a derived busyness proxy `transactions_per_space`.
    """
    return (
        df.group_by("location")
        .agg([
            pl.col("x").min().alias("min_x"),
            pl.col("y").min().alias("min_y"),
            pl.col("rate").median().alias("rate"),
            pl.col("tx_per_meter").sum().alias("tot_transactions"),
            pl.col("spaces_per_meter").sum().alias("tot_spaces"),
            pl.col("dist_m").min().alias("min_dist_m"),
            pl.col("dist_m").median().alias("median_dist_m"),
            pl.col("meter_id").n_unique().alias("n_meters"),
        ])
        .with_columns(
            (pl.col("tot_transactions") / pl.col("tot_spaces")).alias("transactions_per_space"),
            pl.lit(period).alias("period")
        )
    )

def minmax_norm(col: str) -> pl.Expr:
    """
    Create a Polars expression that min-max normalizes a column to the range [0, 1].

    Normalization formula:
        (x - min(x)) / (max(x) - min(x))

    Edge case:
    If max(x) == min(x), the denominator becomes 0 (no variation). In that case,
    this function returns 0.5 for all rows to represent a neutral midpoint.

    Parameters
    col : str
        Column name to normalize.

    Returns
    pl.Expr
        Polars expression producing a normalized column in [0, 1].
    """
    denom = pl.col(col).max() - pl.col(col).min()
    return (
        pl.when(denom == 0)
        .then(pl.lit(0.5))
        .otherwise((pl.col(col) - pl.col(col).min()) / denom)
    )

def add_score(df, w_price=0.3, w_busy=0.2, w_dist=0.5):
    """
    Add normalized features and a preference-weighted score, then return a ranked table.

    The score is a weighted sum of three normalized components (lower is better):
        score = w_price * price_norm
              + w_busy  * busy_norm
              + w_dist  * dist_norm

    Where:
    - price_norm is min-max normalized `rate`
    - busy_norm  is min-max normalized `transactions_per_space`
    - dist_norm  is min-max normalized `min_dist_m`

    Parameters
    df : pl.DataFrame
        Location-level summary table containing columns:
        ['rate', 'transactions_per_space', 'min_dist_m'] at minimum.
    w_price : float, default 0.3
        Weight for normalized price (rate).
    w_busy : float, default 0.2
        Weight for normalized busyness (transactions per space).
    w_dist : float, default 0.5
        Weight for normalized distance (minimum distance in meters).

    Returns
    pl.DataFrame
        Input table with added columns:
        ['price_norm', 'busy_norm', 'dist_norm', 'score'],
        sorted ascending by `score` (best recommendations first).
    """
    return (
        df.with_columns([
            minmax_norm("rate").alias("price_norm"),
            minmax_norm("transactions_per_space").alias("busy_norm"),
            minmax_norm("min_dist_m").alias("dist_norm"),
        ])
        .with_columns(
            (w_price*pl.col("price_norm") + w_busy*pl.col("busy_norm") + w_dist*pl.col("dist_norm")).alias("score")
        )
        .sort("score")
    )


def recommendation_bar(df, title, width=330, height=350):
    """
    Create a bar chart of ranked parking recommendations.

    Parameters
    df : pandas.DataFrame
        Ranked recommendation table containing at least:
        ['location', 'score', 'rate', 'min_x', 'min_y',
         'transactions_per_space', 'tot_transactions', 'tot_spaces',
         'min_dist_m', 'median_dist_m', 'n_meters'].
    title : str
        Chart title.
    width : int, default 330
        Chart width in pixels.
    height : int, default 350
        Chart height in pixels.

    Returns
    alt.Chart
        Altair bar chart sorted by score (lower = better).
    """
    return (
        alt.Chart(df)
        .mark_bar()
        .encode(
            x=alt.X("score:Q", title="Score (lower = better)"),
            y=alt.Y(
                "location:N",
                sort=alt.SortField(field="score", order="ascending"),
                title="Location"
            ),
            color=alt.Color("score:Q", title="Score"),
            tooltip=[
                "location",
                alt.Tooltip("rate:Q", format=".2f", title="Rate"),
                alt.Tooltip("score:Q", format=".2f", title="Score"),
                alt.Tooltip("min_x:Q", format=".6f", title="Closest x"),
                alt.Tooltip("min_y:Q", format=".6f", title="Closest y"),
                alt.Tooltip("transactions_per_space:Q", format=".3f", title="Tx per space (busyness)"),
                alt.Tooltip("tot_transactions:Q", format=",.0f", title="Total transactions"),
                alt.Tooltip("tot_spaces:Q", format=",.0f", title="Total spaces"),
                alt.Tooltip("min_dist_m:Q", format=".2f", title="Min dist (m)"),
                alt.Tooltip("median_dist_m:Q", format=".2f", title="Median dist (m)"),
                alt.Tooltip("n_meters:Q", format=",.0f", title="# meters"),
            ]
        )
        .properties(width=width, height=height, title=title)
    )



# build location summaries for different time slices
loc_m = location_summary(morning_analysis, "Morning")
loc_a = location_summary(afternoon_analysis, "Afternoon")
    
# user preference example: prioritize availability heavily
w_price, w_busy, w_dist = 0.2, 0.7, 0.1

# ranked recommendations for each period
rank_m = add_score(loc_m, w_price, w_busy, w_dist)
rank_a = add_score(loc_a, w_price, w_busy, w_dist)

# take top-N and convert to pandas for Altair plotting
m_top = rank_m.head(10).to_pandas()
a_top = rank_a.head(10).to_pandas()

# usage
c1 = recommendation_bar(m_top, "8AM - 2PM: Recommended locations")
c2 = recommendation_bar(a_top, "2PM - 6PM: Recommended locations")

(c1 | c2).resolve_scale(x="shared", color="shared")

alt.HConcatChart(...)